In [1]:
from graph import app 
import pandas as pd
import os
from tqdm.auto import tqdm
import torch
import time 
from Configuration.models import models #import models to evaluate
from dotenv import load_dotenv
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_neo4j import Neo4jGraph  # for schema
from langchain_ollama import ChatOllama


2026-04-02T17:24:26.517614 [info     ] Deleted old log file: C:\Users\olgag\Desktop\agentic-memory\venv\Lib\site-packages\logs\2026-03-24_22-32-18.log [cognee.shared.logging_utils]

2026-04-02T17:24:32.175900 [warning  ] From version 0.5.0 onwards, Cognee will run with multi-user access control mode set to on by default. Data isolation between different users and datasets will be enforced and data created before multi-user access control mode was turned on won't be accessible by default. To disable multi-user access control mode and regain access to old data set the environment variable ENABLE_BACKEND_ACCESS_CONTROL to false before starting Cognee. For more information, please refer to the Cognee documentation. [cognee.shared.logging_utils]

2026-04-02T17:24:32.175900 [info     ] Log file created at: C:\Users\olgag\Desktop\agentic-memory\venv\Lib\site-packages\logs\2026-04-02_20-24-25.log [cognee.shared.logging_utils] log_file=C:\Users\olgag\Desktop\agentic-memory\venv\Lib\site-packag

[Cognee] Initializing Memory at: c:/Users/olgag/Desktop/agentic-memory/CyVerACT/extended_workflow/.cognee_data
[Cognee] Ready !
[Hybrid Memory Manager] Cognee Connected.
[Cypher Generator] Hybrid Memory Loaded
[Cognee] Initializing Memory at: c:/Users/olgag/Desktop/agentic-memory/CyVerACT/extended_workflow/.cognee_data
[Cognee] Ready !
[Hybrid Memory Manager] Cognee Connected.
[CyVer Evaluator] Hybrid Memory Loaded
[Cognee] Initializing Memory at: c:/Users/olgag/Desktop/agentic-memory/CyVerACT/extended_workflow/.cognee_data
[Cognee] Ready !
[Hybrid Memory Manager] Cognee Connected.
[Cypher Corrector] Hybrid Memory Loaded


TESTING WITH MOVIES DB

In [4]:
from graph import app 
import pandas as pd
import os
from tqdm.auto import tqdm
import time
from dotenv import load_dotenv
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from Memory.hybrid_manager import HybridMemoryManager

load_dotenv(override=True)

print("System Initialization")

if not os.getenv("HUGGINGFACE_API_KEY"):
    print("ERROR: HUGGINGFACE_API_KEY missing.")
elif not os.getenv("NEO4J_URI"):
    print("ERROR: NEO4J URI missing.")
else:
    print("All API Keys found.")

System Initialization
All API Keys found.


In [3]:
models_to_test = [
    {"name": "meta-llama/Meta-Llama-3.1-8B-Instruct", "temperature": 0.0},
    {"name": "meta-llama/Meta-Llama-3.1-8B-Instruct", "temperature": 0.3},
    {"name": "meta-llama/Meta-Llama-3.1-8B-Instruct", "temperature": 0.9},
    {"name": "meta-llama/Llama-3.2-3B-Instruct", "temperature": 0.0},
]

In [4]:
custom_dataset = [
    "Find all actors who acted in more than 3 movies",
    "Find actors and how many distinct movies they acted in",
    "Find movies that have more than 3 actors",
    "Find actors who acted in movies directed by Ron Howard",
    "Find actors who acted in exactly one movie",
    "Find all directors who directed at least 2 movies",
    "Find all people who both acted in and directed movies",
    "Find all actors who acted in The Matrix",
    "Find how many movies were released per year",
    "Return each movie along with its director"
]

In [5]:
def create_model(model_name, temperature):
    hf_endpoint = HuggingFaceEndpoint(
        repo_id=model_name,
        task="text-generation",
        max_new_tokens=512,
        temperature=temperature,
        huggingfacehub_api_token=os.getenv("HUGGINGFACE_API_KEY")
    )
    return ChatHuggingFace(llm=hf_endpoint)


def detect_loop(query_history):
    # Detect repeated identical queries
    if len(query_history) < 2:
        return False
    return query_history[-1] == query_history[-2]

In [6]:
dataset = [
    "Find all actors who acted in more than 3 movies",
    "Find actors and how many distinct movies they acted in",
    "Find movies that have more than 3 actors",
    "Find actors who acted in movies directed by Ron Howard",
    "Find actors who acted in exactly one movie",
    "Find all directors who directed at least 2 movies",
    "Find all people who both acted in and directed movies",
    "Find all actors who acted in The Matrix",
    "Find how many movies were released per year",
    "Return each movie along with its director"
]

In [ ]:
all_results = []

for model_config in models_to_test:

    model_name = model_config["name"]
    temp = model_config["temperature"]

    print("\n" + "="*80)
    print(f"🚀 Running Model: {model_name} | Temp: {temp}")
    print("="*80)

    # RESET MEMORY PER MODEL (CRITICAL)
    memory = HybridMemoryManager(reset_db=True)

    model = create_model(model_name, temp)

    config = {
        'configurable': {
            'model': model,
            'model_name': model_name,
            'total_attempts': 5,
            'attempts_w_filtered': 3,
        },
        'recursion_limit': 50
    }

    inputs = {
        'database_url': os.getenv("NEO4J_URI"),
        'database_name': "movies",
        'database_user': os.getenv("NEO4J_USERNAME"),
        'database_password': os.getenv("NEO4J_PASSWORD")
    }

    for i, question in enumerate(tqdm(custom_dataset, desc=f"{model_name}-{temp}")):

        print('\n' + '#' * 60)
        print(f"QUERY {i+1}: {question}")
        print('#' * 60)

        inputs['question'] = question

        start_time = time.time()

        try:
            result = app.invoke(inputs, config=config)

            duration = time.time() - start_time

            # Extract metadata if available
            attempts = result.get("attempts", None)
            corrections = result.get("corrections", None)
            cypher_query = result.get("cypher", "")

            all_results.append({
                "model": model_name,
                "temperature": temp,
                "question": question,
                "success": True,
                "attempts": attempts,
                "corrections": corrections,
                "execution_time": duration,
                "cypher": cypher_query
            })

            print(f"✅ Success | Time: {duration:.2f}s")

        except Exception as e:
            duration = time.time() - start_time

            all_results.append({
                "model": model_name,
                "temperature": temp,
                "question": question,
                "success": False,
                "error": str(e),
                "execution_time": duration
            })

            print(f"❌ Failed | Error: {e}")

        time.sleep(5)  # avoid rate limits


🚀 Running Model: meta-llama/Meta-Llama-3.1-8B-Instruct | Temp: 0.0
[Cognee] Initializing Memory at: c:/Users/olgag/Desktop/agentic-memory/CyVerACT/extended_workflow/.cognee_data
[Cognee] Manual Reset: Pruning data...



2026-03-24T20:33:20.275973 [info     ] Database deleted successfully. [cognee.shared.logging_utils]

Storage manager absolute path: C:\Users\olgag\Desktop\agentic-memory\venv\Lib\site-packages\cognee\.cognee_cache

Deleting cache...             

✓ Cache deleted successfully! 


[Cognee] Ready !
[Hybrid Memory Manager] Cognee Connected.


meta-llama/Meta-Llama-3.1-8B-Instruct-0.0:   0%|          | 0/10 [00:00<?, ?it/s]


############################################################
QUERY 1: Find all actors who acted in more than 3 movies
############################################################
Schema filtering Agent: START
[Cognee] Searching rules for: Find all actors who acted in more than 3 movies...
User 2f9628d9-ad7b-4bb4-89cc-0daf2d9cc92a has registered.


...


2026-03-24T21:06:59.380532 [warning  ] Search attempt on an empty knowledge graph [cognee.shared.logging_utils]

2026-03-24T21:06:59.414628 [warning  ] Search attempt on an empty knowledge graph [GraphCompletionRetriever]
Our latest automated health check on model 'meta-llama/Llama-3.2-3B-Instruct' for provider 'hyperbolic' did not complete successfully.  Inference call might fail.

Our latest automated health check on model 'meta-llama/Llama-3.2-3B-Instruct' for provider 'hyperbolic' did not complete successfully.  Inference call might fail.


[Cognee] Found 1 rules.
 Rule 1: - - Verify relationship direction: match the correct -> or <- on :ACTED_IN; if unknown, use undirected or test both.
- Bind movie nodes before aggregating: MATCH (a:Actor)-[:ACTED_IN]->(m:Movie) then aggregate.
- Use COUNT(DISTINCT m) to count distinct movies and avoid inflated counts from multiple paths.
- Aggregate only after a WITH that carries grouping keys: WITH a, m ... WITH a, COUNT(DISTINCT m) AS movies.
- If using OPTIONAL MATCH, handle nulls (COUNT(DISTINCT m) yields 0 for no matches).
- Prevent duplicate matches from multiple patterns: dedupe with DISTINCT or collect(m) then size(collect).
- Confirm schema: correct labels/casing (Actor vs Person), exact property names (title/name) and relationship type spelling.
- Ensure variables used in RETURN/WITH are actually bound in previous MATCH/OPTIONAL MATCH.
❌ Failed | Error: (Request ID: Root=1-69c2fcfe-07d0667136ddeac14ff3805a;6ee6ed6d-fde5-402c-8233-170071beb2e2)

Bad request:
{'code': 40301, 'm


2026-03-24T21:07:17.758790 [warning  ] Search attempt on an empty knowledge graph [cognee.shared.logging_utils]

2026-03-24T21:07:17.778448 [warning  ] Search attempt on an empty knowledge graph [GraphCompletionRetriever]
Our latest automated health check on model 'meta-llama/Llama-3.2-3B-Instruct' for provider 'hyperbolic' did not complete successfully.  Inference call might fail.

Our latest automated health check on model 'meta-llama/Llama-3.2-3B-Instruct' for provider 'hyperbolic' did not complete successfully.  Inference call might fail.


[Cognee] Found 1 rules.
 Rule 1: - - Ensure you aggregate correctly: bring the movie node through WITH before HAVING (e.g., WITH m, COUNT(DISTINCT a) AS actorsCount HAVING actorsCount > 3).
- Use COUNT(DISTINCT actor) to avoid overcounting when multiple paths or duplicate relationships exist.
- Prefer DISTINCT in the MATCH when different paths produce the same actor (or use COUNT(DISTINCT)).
- Check relationship direction: use the correct arrow for actor→movie (or movie←actor) as your schema defines; wrong direction returns no matches.
- Verify exact relationship type and label names (case-sensitive): :Movie, :Actor, :ACTED_IN vs :PLAYED_IN, etc.
- Confirm nodes actually have the expected labels/properties; absent labels yield empty results.
- Bring only needed variables through WITH to avoid losing references (e.g., include m when aggregating actors).
- If duplicates persist, inspect graph for parallel relationships or intermediate nodes causing multiplicity and use DISTINCT or collap

KeyboardInterrupt: 

In [8]:
df = pd.DataFrame(all_results)

print("\n📊 Summary by Model:")
summary = df.groupby(["model", "temperature"]).agg({
    "success": "mean",
    "execution_time": "mean"
}).rename(columns={
    "success": "success_rate",
    "execution_time": "avg_time"
})

print(summary)

df.to_csv("experiment_results.csv", index=False)
print("\n💾 Results saved to experiment_results.csv")


📊 Summary by Model:
                                                   success_rate   avg_time
model                                 temperature                         
meta-llama/Llama-3.2-3B-Instruct      0.0                   0.0  14.237178
meta-llama/Meta-Llama-3.1-8B-Instruct 0.0                   1.0  58.512857
                                      0.3                   1.0  83.347656
                                      0.9                   1.0  41.244687

💾 Results saved to experiment_results.csv


END OF TESTING WITH MOVIES DB

In [2]:
import cognee

# Remove all user data (datasets, documents)
await cognee.prune.prune_data()

# Remove all system data (graph, vectors, metadata, cache)
await cognee.prune.prune_system(metadata=True)


2026-04-02T12:12:07.416969 [info     ] Database deleted successfully. [cognee.shared.logging_utils]

Storage manager absolute path: C:\Users\olgag\Desktop\agentic-memory\venv\Lib\site-packages\cognee\.cognee_cache

Deleting cache...             

✓ Cache deleted successfully! 


In [2]:
# Loading .env
load_dotenv(override=True) 

print("System Initialization")

# Key Checksss
if not os.getenv("OPENAI_API_KEY"):
    print("ERROR: OPENAI_API_KEY missing.")
#elif not os.getenv("GROQ_API_KEY"):
    #print("ERROR: GROQ_API_KEY missing. Check your .env file.")
elif not os.getenv("HUGGINGFACE_API_KEY"):
    print("ERROR: HUGGINGFACE_API_KEY missing.")
elif not os.getenv("NEO4J_URI"):
    print("ERROR: NEO4J URI missing.")
else:
    print("All API Keys found.")

# HYBRID MEMORY Init
memory = None
try:
    from Memory.hybrid_manager import HybridMemoryManager
    print("\nLoading Hybrid Memory Manager...")
    
    memory = HybridMemoryManager()
    
    # Memory Check
    if memory.cognee:
        print("Cognee (Semantic): ACTIVE")
    else:
        print("Cognee (Semantic): INACTIVE ")
        
    if memory.graphiti:
        print("Graphiti Episodic: ACTIVE")
    else:
        print("Graphiti Episodic: INACTIVE")

except ImportError:
    print("ERROR: Could not import HybridMemoryManager.")
except Exception as e:
    print(f"Error initializing Memory: {e}")

# Model Settings
#from langchain_community.chat_models import ChatOllama

#from langchain_openai import ChatOpenAI
#model_name = "llama3.2"
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
#model_name = "meta-llama/Llama-3.2-1B-Instruct"
#model_name = "Qwen/Qwen2.5-3B-Instruct"
#model_name = 'gpt-4o'
#model_name = "llama-3.3-70b-versatile"
#model_name = "llama-3.1-8b-instant"
# model_name = "qwen/qwen3-32b"
#model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# print(f"\nInitializing Groq model: {model_name}...")

# try:
#     model = ChatGroq(
#         model=model_name,
#         temperature=0.9, # change later?
#         api_key=os.getenv("GROQ_API_KEY")
#     )
#     print(f"Model {model_name} loaded successfully.")
    
# except Exception as e:
#     print(f"Error loading model: {e}")

print(f"\nInitializing Hugging Face model: {model_name}...")

try:
    hf_endpoint = HuggingFaceEndpoint(
        repo_id=model_name, 
        task="text-generation",
        max_new_tokens=512,
        temperature=0.1, 
        huggingfacehub_api_token=os.getenv("HUGGINGFACE_API_KEY")
    )

    model = ChatHuggingFace(llm=hf_endpoint)
    print(f"Model {model_name} loaded successfully.")
    
except Exception as e:
    print(f"Error loading model: {e}")

# try:
#     #model = ChatOpenAI(model=model_name, temperature=0)
#     model = ChatOllama(model=model_name, temperature=0.6)
#     print(f"\nModel {model_name} loaded successfully.")
# except Exception as e:
#     print(f"Error loading model: {e}")

# Config Settings
config = {
    'configurable': {
        'model': model, 
        'model_name': model_name, 
        #'thread_id': 'demo_1',
        'total_attempts': 5,          # total 
        'attempts_w_filtered': 3,     # with filtered
    },
    'recursion_limit': 50
}

# Setting up inputs for next cell
inputs = {
    'database_url': os.getenv("NEO4J_URI"),           
    'database_name': "movies",                        
    'database_user': os.getenv("NEO4J_USERNAME"),    
    'database_password': os.getenv("NEO4J_PASSWORD")  
}

print("\nSetup Complete! Ready for our agent!")

System Initialization
All API Keys found.

Loading Hybrid Memory Manager...
[Cognee] Initializing Memory at: c:/Users/olgag/Desktop/agentic-memory/CyVerACT/extended_workflow/.cognee_data
[Cognee] Ready !
[Hybrid Memory Manager] Cognee Connected.
Cognee (Semantic): ACTIVE
Graphiti Episodic: INACTIVE

Initializing Hugging Face model: meta-llama/Meta-Llama-3.1-8B-Instruct...
Model meta-llama/Meta-Llama-3.1-8B-Instruct loaded successfully.

Setup Complete! Ready for our agent!


In [3]:
custom_dataset = [
    "Find all actors who acted in more than 3 movies",
    "Find actors and how many distinct movies they acted in",
    "Find movies that have more than 3 actors",
    "Find actors who acted in movies directed by Ron Howard",
    "Find actors who acted in exactly one movie",
    "Find all directors who directed at least 2 movies",
    "Find all people who both acted in and directed movies",
    "Find all actors who acted in The Matrix",
    "Find how many movies were released per year",
    "Return each movie along with its director"
]

print("Starting evaluation for custom_dataset on Movies DB...")
results_log = [] 

for i, question in enumerate(tqdm(custom_dataset, desc="Evaluating Queries")):
    print('\n' + '#' * 60)
    print(f" QUERY {i+1}: {question}")
    print('#' * 60)
    
    inputs['question'] = question
    
    try:
        result = app.invoke(inputs, config=config)
        answer = result.get('answer')
        
        print('\n' + '='*50)
        print(f"Final Agent Answer: {answer}")
        print('='*50 + '\n')
        
        results_log.append({"question": question, "status": "Success", "answer": answer})
        
    except Exception as e:
        print(f"\n Execution Failed: {e}")
        results_log.append({"question": question, "status": "Failed", "error": str(e)})
    
    if i < len(custom_dataset) - 1:
        print("Waiting 10s for API rate limits...")
        time.sleep(10)

print("\n Evaluation Complete!")

Starting evaluation for custom_dataset on Movies DB...


Evaluating Queries:   0%|          | 0/10 [00:00<?, ?it/s]


############################################################
 QUERY 1: Find all actors who acted in more than 3 movies
############################################################
Schema filtering Agent: START
[Cognee] Searching rules for: Find all actors who acted in more than 3 movies...
User 1254193f-247f-4226-8276-2cb273e2db3d has registered.



2026-04-02T17:25:56.029313 [warning  ] Search attempt on an empty knowledge graph [cognee.shared.logging_utils]

2026-04-02T17:25:56.059820 [warning  ] Search attempt on an empty knowledge graph [GraphCompletionRetriever]


[Cognee] No meaningful results found.
Cypher Generator: START
CyVer Evaluator: START
CyVer Evaluation of MATCH (p:Person)-[r:ACTED_IN]->(m:Movie) WITH p, COUNT(r) as count GROUP BY p HAVING count > 3 RETURN p:
[CyVer Evaluator] Errors detected: 1
CyVer Errors from generated query:
- Neo.ClientError.Statement.SyntaxError: Invalid input 'GROUP': expected ',', 'ORDER BY', 'CALL', 'CREATE', 'LOAD CSV', 'DELETE', 'DETACH', 'FILTER', 'FINISH', 'FOREACH', 'INSERT', 'LET', 'LIMIT', 'MATCH', 'MERGE', 'NEXT', 'NODETACH', 'OFFSET', 'OPTIONAL', 'REMOVE', 'RETURN', 'SET', 'SKIP', 'UNION', 'UNWIND', 'USE', 'WHERE', 'WITH' or <EOF> (line 1, column 76 (offset: 75))


[CyVer Evaluator] Storing mistake in Hybrid Memory...
[Cognee] Learning lesson...


...

[Cognee] Found 1 rules.
 Rule 1: - - Use WITH before aggregates: perform COUNT/other aggregation in a WITH clause, then filter with WHERE; do not use SQL-style GROUP BY/HAVING.
- Relationship direction matters: match the actual arrow for DIRECTED (e.g., (d)-[:DIRECTED]->(m) or reverse as your graph defines); wrong direction returns no results.
- Check schema: confirm correct labels and property names (e.g., :Movie, :Person, title, name); misspelled labels/properties cause empty results.
- Avoid duplicates: return unique pairs with DISTINCT (e.g., RETURN DISTINCT m, d) or constrain your pattern to produce one row per movie-director.
Cypher Generator: START
CyVer Evaluator: START
CyVer Evaluation of MATCH (m:Movie)-[d:DIRECTED]->(p:Person) RETURN m, p:
[CyVer Evaluator] Errors detected: 1
CyVer Errors from generated query:
- WrongDirectionPathWarning: The specified path (:Movie)-[:DIRECTED]->(:Person) has the opposite direction in the knowledge graph schema. The path (m:Movie)-[d:DIRECTE


2026-04-02T17:34:35.603827 [info     ] Pipeline run started: `29e492e0-7428-5f82-9eb9-13a661069d10` [run_tasks_with_telemetry()]

2026-04-02T17:34:35.905352 [info     ] Coroutine task started: `resolve_data_directories` [run_tasks_base]

2026-04-02T17:34:36.215532 [info     ] Coroutine task started: `ingest_data` [run_tasks_base]

2026-04-02T17:34:36.702608 [info     ] Coroutine task completed: `ingest_data` [run_tasks_base]

2026-04-02T17:34:37.017990 [info     ] Coroutine task completed: `resolve_data_directories` [run_tasks_base]

2026-04-02T17:34:37.337164 [info     ] Pipeline run completed: `29e492e0-7428-5f82-9eb9-13a661069d10` [run_tasks_with_telemetry()]

2026-04-02T17:34:39.750313 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapter]

2026-04-02T17:34:40.027029 [info     ] Pipeline run started: `3708c309-e1a2-583d-af6f-9d2defd99790` [run_tasks_with_telemetry()]

2026-04-02T17:34:40.321226 [info     ] Coroutine task started: `c

[Cognee] Memory Updated!
[CyVer Evaluator] Lesson Saved!
[Cypher Corrector] Searching procedural memory...
[Cognee] Searching rules for: 
            Cypher mistake case.
            Errors: WrongDirectionPathWarning: The specified path (:Movie)-[:DIRECTED]->(:Person) has the opposite direction in the knowledge graph schema. The path (m:Movie)-[d:DIRECTED]->(p:Person) in the Cypher statement should be changed to  (m:Movie)<-[d:DIRECTED]-(p:Person).
            Faulty query: MATCH (m:Movie)-[d:DIRECTED]->(p:Person) RETURN m, p
            ...



2026-04-02T17:36:01.754766 [info     ] Vector collection retrieval completed: Retrieved distances from 4 collections in 0.34s [cognee.shared.logging_utils]

2026-04-02T17:36:01.774256 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-04-02T17:36:01.985877 [info     ] ID-filtered retrieval: 12 nodes and 9 edges in 0.21s [Neo4jAdapter]

2026-04-02T17:36:01.989871 [info     ] Graph projection completed: 12 nodes, 9 edges in 0.00s [CogneeGraph]


[Cognee] Found 1 rules.
 Rule 1: - - Reverse relationship direction for DIRECTED: match directors as (m:Movie)<-[d:DIRECTED]-(p:Person).
- Always use WITH for aggregation: compute aggregates with WITH (e.g., WITH p, COUNT(r) AS cnt) then filter using WHERE — do not use SQL-style GROUP BY/HAVING.
- Deduplicate explicitly: use DISTINCT or aggregate (COLLECT / COUNT DISTINCT) when duplicate nodes/rows may appear.
- Use valid schema identifiers and defined values: use correct labels/properties and supply parameters or literal lists for property values (never use an undefined bare identifier like LIST).
[Memory] Found past mistakes for similar queries!- - Reverse relationship direction for DIRECTED: match directors as (m:Movie)<-[d:DIRECTED]-(p:Person).
- Always use WITH for aggregation: compute aggregates with WITH (e.g., WITH p, COUNT(r) AS cnt) then filter using WHERE — do not use SQL-style GROUP BY/HAVING.
- Deduplicate explicitly: use DISTINCT or aggregate (COLLECT / COUNT DISTINCT) wh

In [ ]:
# Running Agent + Updating Episodic Memory

# Question 
#question = "Give me all the directors who acted in more than 3 movies"
question = "Find actors who acted in more movies than the average actor"
#question = "Which studio distributed the movie The Matrix" 
# For later testing ... 
#question = "Give me all actors and the films they acted in."

print(f"User Asking: '{question}'")

# Updating inputs
inputs['question'] = question

try:
    '''cypher_generator calls HybridManager to get rules from Cognee and episodic memory from Graphiti'''
    result = app.invoke(inputs, config=config)
    
    # Formatting
    print('\n' + '='*50)
    print(f"Final Agent Answer: {result.get('answer')}")
    print('='*50 + '\n')

except Exception as e:
    print(f"Execution Failed: {e}")

In [ ]:
from neo4j import GraphDatabase
# Loading .env
load_dotenv(override=True) 
print("System Initialization")

# Key Checks
if not os.getenv("OPENAI_API_KEY"):
    print("ERROR: OPENAI_API_KEY missing.")
elif not os.getenv("HUGGINGFACE_API_KEY"):
    print("ERROR: HUGGINGFACE_API_KEY missing.")
elif not os.getenv("NEO4J_URI"):
    print("ERROR: NEO4J URI missing (Local Memory).")
elif not os.getenv("SANDBOX_URI"):
    print("ERROR: SANDBOX_URI missing! Check your .env file.")
else:
    print("✅ All API Keys & Database URIs found.")

# ---------------------------------------------------------
# ΟΡΙΣΜΟΣ ΜΕΤΑΒΛΗΤΩΝ SANDBOX (Αυτό έλειπε!)
# ---------------------------------------------------------
sandbox_uri = os.getenv("SANDBOX_URI")
sandbox_user = os.getenv("SANDBOX_USERNAME")
sandbox_pass = os.getenv("SANDBOX_PASSWORD")

# ---------------------------------------------------------
# HYBRID MEMORY Init
# ---------------------------------------------------------
memory = None
try:
    from Memory.hybrid_manager import HybridMemoryManager
    print("\nLoading Hybrid Memory Manager...")
    
    memory = HybridMemoryManager()
    
    if memory.cognee:
        print("✅ Cognee (Semantic): ACTIVE")
    else:
        print("❌ Cognee (Semantic): INACTIVE ")
        
    if memory.graphiti:
        print("✅ Graphiti Episodic: ACTIVE")
    else:
        print("❌ Graphiti Episodic: INACTIVE")

except ImportError:
    print("ERROR: Could not import HybridMemoryManager.")
except Exception as e:
    print(f"Error initializing Memory: {e}")

# ---------------------------------------------------------
# Model Settings
#---------------------------------------------------------
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
print(f"\nInitializing Hugging Face model: {model_name}...")

try:
    hf_endpoint = HuggingFaceEndpoint(
        repo_id=model_name, 
        task="text-generation",
        max_new_tokens=512,
        temperature=0.3, 
        huggingfacehub_api_token=os.getenv("HUGGINGFACE_API_KEY")
    )

    model = ChatHuggingFace(llm=hf_endpoint)
    print(f"✅ Model {model_name} loaded successfully.")
    
except Exception as e:
    print(f"❌ Error loading model: {e}")

# ---------------------------------------------------------
# Config Settings
# ---------------------------------------------------------
config = {
    'configurable': {
        'model': model, 
        'model_name': model_name, 
        'total_attempts': 5,          # total 
        'attempts_w_filtered': 3,     # with filtered
    },
    'recursion_limit': 50
}

# ---------------------------------------------------------
# Schema Fetching (LangChain Rich Format)
# ---------------------------------------------------------
print("\nFetching RICH Schema directly from Neo4j Sandbox...")
accurate_schema = """Node properties:
- **Client** {id: STRING, name: STRING}
- **Bank** {id: STRING, name: STRING}
- **Merchant** {id: STRING, name: STRING}
- **Mule** {id: STRING, name: STRING}
- **CashIn** {amount: FLOAT, fraud: BOOLEAN, step: INTEGER, id: STRING, globalStep: INTEGER, ts: INTEGER}
- **CashOut** {amount: FLOAT, fraud: BOOLEAN, step: INTEGER, id: STRING, globalStep: INTEGER, ts: INTEGER}
- **Debit** {amount: FLOAT, fraud: BOOLEAN, step: INTEGER, id: STRING, globalStep: INTEGER, ts: INTEGER}
- **Payment** {amount: FLOAT, fraud: BOOLEAN, step: INTEGER, id: STRING, globalStep: INTEGER, ts: INTEGER}
- **Transfer** {amount: FLOAT, fraud: BOOLEAN, step: INTEGER, id: STRING, globalStep: INTEGER, ts: INTEGER}
- **Transaction** {amount: FLOAT, fraud: BOOLEAN, step: INTEGER, id: STRING, globalStep: INTEGER, ts: INTEGER}
- **Email** {email: STRING}
- **SSN** {ssn: STRING}
- **Phone** {phoneNumber: STRING}

Relationship properties:
(None)

The relationships:
(:Client)-[:PERFORMED]->(:CashIn)
(:Client)-[:PERFORMED]->(:CashOut)
(:Client)-[:PERFORMED]->(:Debit)
(:Client)-[:PERFORMED]->(:Payment)
(:Client)-[:PERFORMED]->(:Transfer)
(:Client)-[:PERFORMED]->(:Transaction)
(:Client)-[:HAS_EMAIL]->(:Email)
(:Client)-[:HAS_SSN]->(:SSN)
(:Client)-[:HAS_PHONE]->(:Phone)
(:Transaction)-[:TO]->(:Client)
(:Transaction)-[:TO]->(:Mule)
(:Transaction)-[:TO]->(:Merchant)
(:Transaction)-[:TO]->(:Bank)
"""

# ---------------------------------------------------------
# Inputs setup
# ---------------------------------------------------------
inputs = {
    'database_url': sandbox_uri,           
    'database_name': "neo4j",                        
    'database_user': sandbox_user,    
    'database_password': sandbox_pass,
    'schema': accurate_schema 
}

print("\n🚀 Setup Complete! Ready for our agent!") 

# ---------------------------------------------------------
# Custom Dataset & Execution
# ---------------------------------------------------------
custom_dataset = [
    "Count the total number of transactions where the fraud property is true.",
    "Find all clients who performed a transaction that was then sent TO a merchant. Return the client's ID and the merchant's ID. Limit to 5.",
    "Which specific bank has processed the highest total amount of money through fraudulent transactions? Return the bank ID and the total amount.",
    #"Find pairs of distinct clients who share the exact same email address. Return the email and both of their IDs.",
    #"Which merchant has received the highest ratio of fraudulent transactions compared to their total received transactions? Return the merchant ID and the ratio."
]

print("Starting evaluation for custom_fraud_dataset on Sandbox ...")
results_log = [] 

for i, question in enumerate(tqdm(custom_dataset, desc="Evaluating Queries")):
    print('\n' + '#' * 60)
    print(f"🎯 QUERY {i+1}: {question}")
    print('#' * 60)
    
    inputs['question'] = question
    
    try:
        # Βεβαιώσου ότι η μεταβλητή 'app' είναι imported στο notebook σου!
        result = app.invoke(inputs, config=config)
        answer = result.get('answer')
        
        print('\n' + '='*50)
        print(f"✅ Final Agent Answer: {answer}")
        print('='*50 + '\n')
        
        results_log.append({"question": question, "status": "Success", "answer": answer})
        
    except Exception as e:
        print(f"\n❌ Execution Failed: {e}")
        results_log.append({"question": question, "status": "Failed", "error": str(e)})
    
    if i < len(custom_dataset) - 1:
        print("⏳ Waiting 10s for API rate limits...")
        time.sleep(10)

print("\n🎉 Evaluation Complete!")

In [ ]:
import nltk

# Dowloading these for NLTK
print("Downloading NLTK data...")
nltk.download('stopwords')
nltk.download('punkt')      
nltk.download('punkt_tab')  
print("Done! You can now run your agent.")

# Evaluate models

In [3]:
import os
import pandas as pd
from tqdm import tqdm
import torch

# With limit for testing
def evaluate_model(model_name, model, dataset, limit=None):
    
    # Loading dataset
    if dataset == 'sustain':
        df = pd.read_excel('../data/text2cypher-sustaingraph/sustaintest.xlsx', sheet_name='Queries')
        subfolder = 'SustainDataset'
    elif dataset == 'benchmark2025':
        df = pd.read_excel('../data/text2cypher-2025v1/test.xlsx')
        df = df[df['url'].notna()].reset_index(drop=True)
        subfolder = 'BenchmarkDataset2025'
    elif dataset == 'benchmark2024':
        df = pd.read_excel('../data/text2cypher-2024v1/test.xlsx')
        df = df[df['url'].notna()].reset_index(drop=True)
        subfolder = 'BenchmarkDataset2024'
    else:
        raise ValueError("Invalid dataset name!")
        
    if limit is not None:
        df = df.head(limit)
        print(f"\n TEST: Only first {limit} questions!\n")

    # For outputs
    output_dir = f"Outputs_Evaluation/{subfolder}/{model_name}/"
    os.makedirs(output_dir, exist_ok=True)

    results_path_pkl = os.path.join(output_dir, f"{model_name}.pkl")
    results_path_xlsx = os.path.join(output_dir, f"{model_name}.xlsx")
   
    # Resume ignoring the completed questions 
    if os.path.exists(results_path_xlsx):
        print(f"Found existing progress at {results_path_xlsx}. Resuming...")
        df_results = pd.read_excel(results_path_xlsx)
    else:
        df_results = df.copy() 
        new_cols = ['generated_cypher', 'answer', 'path', 'output', 'cypher_errors_history', 
                    'cypher_errors', 'is_KG_Valid_correct', 'total_tokens', 
                    'completion_tokens', 'prompt_tokens', 'is_KG_Valid_correct_with_full_schema']
        for col in new_cols:
            df_results[col] = None

    # LangGraph config
    config = {
        'configurable': {
            'model': model, 
            'model_name': model_name, 
            'attempts_w_filtered': 3, 
            'total_attempts': 5
        },
        'recursion_limit': 50
    }
    
    print(f'Starting evaluation for {model_name} on {dataset} ...')
    
    # Main Loop
    for i, row in tqdm(df_results.iterrows(), total=df_results.shape[0]):
        if pd.isna(row['generated_cypher']):
            question = row['Question'] if dataset == 'sustain' else row['question']
            
            # Getting credentials for all neo4j dbs
            required_cols = {'database_name', 'url', 'username', 'password'}
            if required_cols.issubset(df_results.columns):
                database_url = row['url']
                database_name = row['database_name']
                database_user = row['username']
                database_password = row['password']
            else:
                database_url = os.getenv('NEO4J_URI')
                database_name = os.getenv('GRAPH_DATABASE_NAME', 'neo4j')
                database_user = os.getenv('NEO4J_USERNAME')
                database_password = os.getenv('NEO4J_PASSWORD')

            inputs = {
                'question': question,
                'database_url': database_url,
                'database_name': database_name,
                'database_user': database_user,
                'database_password': database_password
            }
            
            try:
                result = app.invoke(inputs, config=config)
                
                df_results.at[i, 'generated_cypher'] = result.get('cypher_statement', '')
                df_results.at[i, 'output'] = str(result.get('database_records', ''))
                df_results.at[i, 'path'] = str(result.get('path', [])) 
                df_results.at[i, 'cypher_errors'] = str(result.get('cypher_errors', []))
                df_results.at[i, 'cypher_errors_history'] = str(result.get('cypher_errors_history', []))
                df_results.at[i, 'is_KG_Valid_correct'] = result.get('is_KG_Valid_correct', False)
                df_results.at[i, 'answer'] = result.get('answer', '')
                
                path_list = result.get('path', [])
                valid_correct = result.get('is_KG_Valid_correct', False)
                df_results.at[i, 'is_KG_Valid_correct_with_full_schema'] = ('schema_retriever' in path_list) and valid_correct
                
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                    
                df_results.to_pickle(results_path_pkl)
                df_results.to_excel(results_path_xlsx, index=False)
                
            except Exception as e:
                print(f"Error at index {i}: {e}")
                df_results.at[i, 'generated_cypher'] = "ERROR"
                df_results.at[i, 'answer'] = str(e)
                df_results.to_excel(results_path_xlsx, index=False)
                
    print(f"\nEvaluation Complete! Results saved at: {results_path_xlsx}")
    return df_results

In [7]:
# ONLY n questions my laptop can't take much!
test_results_df = evaluate_model(
    model_name="agent_test", 
    model=model,                      
    dataset="benchmark2024",           
    limit=5                           
)

# Printing results to check 
display(test_results_df[['question', 'generated_cypher', 'is_KG_Valid_correct']])


 TEST: Only first 5 questions!

Starting evaluation for agent_test on benchmark2024 ...


  0%|          | 0/5 [00:00<?, ?it/s]

Schema filtering Agent: START
[Cognee] Searching rules for: Identify the 5 suppliers with the highest average unit price of products supplied....



2026-03-19T21:57:38.920512 [info     ] Vector collection retrieval completed: Retrieved distances from 6 collections in 0.61s [cognee.shared.logging_utils]

2026-03-19T21:57:38.933028 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-03-19T21:57:39.386008 [info     ] ID-filtered retrieval: 66 nodes and 125 edges in 0.45s [Neo4jAdapter]

2026-03-19T21:57:39.403356 [info     ] Graph projection completed: 66 nodes, 125 edges in 0.02s [CogneeGraph]


[Cognee] Found 1 rules.
 Rule 1: - I can’t identify the top 5 suppliers because the provided context contains only movie/genre error logs and no supplier/product/unit-price data. If you can run a query on the supplier/product graph, use (for example):

MATCH (s:Supplier)-[:SUPPLIES]->(p:Product)
RETURN s.name AS supplier, avg(p.unitPrice) AS avgUnitPrice
ORDER BY avgUnitPrice DESC
LIMIT 5;

Provide the supplier/product data or query results and I’ll list the five suppliers.
Cypher Generator: START
CyVer Evaluator: START
CyVer Evaluation of MATCH (s:Supplier)-[:SUPPLIES]->(p:Product) RETURN s.companyName AS supplier, avg(p.unitPrice) AS avgUnitPrice ORDER BY avgUnitPrice DESC LIMIT 5:
Cypher Executor: START
Executing Cypher: MATCH (s:Supplier)-[:SUPPLIES]->(p:Product) RETURN s.companyName AS supplier, avg(p.unitPrice) AS avgUnitPrice ORDER BY avgUnitPrice DESC LIMIT 5
Records found: 5
Cypher Executor: END
Output Generator: START
Agent Answer: I found the following in the database: suppl

...


2026-03-19T22:01:10.761309 [info     ] Vector collection retrieval completed: Retrieved distances from 6 collections in 0.68s [cognee.shared.logging_utils]

2026-03-19T22:01:10.761309 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-03-19T22:01:11.166339 [info     ] ID-filtered retrieval: 75 nodes and 144 edges in 0.41s [Neo4jAdapter]

2026-03-19T22:01:11.182027 [info     ] Graph projection completed: 75 nodes, 144 edges in 0.01s [CogneeGraph]


[Cognee] Found 1 rules.
 Rule 1: - No tweet nodes or tweet IDs appear in the provided context, so I can’t determine the oldest tweet id. If your tweets are in Neo4j, you can run, for example: MATCH (t:Tweet) RETURN t.id ORDER BY t.created_at ASC LIMIT 1
Cypher Generator: START
CyVer Evaluator: START
CyVer Evaluation of MATCH (t:Tweet) RETURN t.id ORDER BY t.created_at ASC LIMIT 1:
Cypher Executor: START
Executing Cypher: MATCH (t:Tweet) RETURN t.id ORDER BY t.created_at ASC LIMIT 1
Records found: 1
Cypher Executor: END
Output Generator: START
Agent Answer: I found the following in the database: t.id: 1335451913203163138
[HybridMemoryManager] Sending Episode to Graphiti...



Completed add_episode in 6426.974296569824 ms
100%|██████████| 5/5 [03:58<00:00, 47.73s/it]

[Graphiti] Episodic memory stored.
[Output Generator] Episodic memory saved successfully!
Output Generator: END

Evaluation Complete! Results saved at: Outputs_Evaluation/BenchmarkDataset2024/agent_test/agent_test.xlsx


,question,generated_cypher,is_KG_Valid_correct
0,Identify the 5 suppliers with the highest aver...,MATCH (s:Supplier)-[:SUPPLIES]->(p:Product) RE...,True
1,List the articles that mention the organizatio...,MATCH (a:Article)-[:MENTIONS]->(:Organization ...,True
2,List the first 3 genres of movies with the low...,MATCH (movie:Movie)-[:IN_GENRE]->(genre:Genre)...,True
3,What is the total number of users who are VIPs...,MATCH (u:User)-[:VIP]->(s:Stream)\nWHERE EXIST...,True
4,"""show me the oldest tweet id""",MATCH (t:Tweet) RETURN t.id ORDER BY t.created...,True


For Gemma 9b add : 


if database_name == 'offshoreleaks':

print(f'Skipping row {i}, to avoid memory error in {database_name}')

continue

In [ ]:
def evaluate_model (model_name, model, dataset):
    ''' 
    Evaluate the selected model on the given dataset, by invoking the graph workflow.
    Save the dataframe with the questions and the generated cypher queries to the Outputs_Evaluation folder.
    Return the dataframe
    '''
    # torch.cuda.empty_cache()#clear gpu cache
    # Get datasets 
    if dataset =='sustain':
        df = pd.read_excel('../data/text2cypher-sustaingraph/sustaintest.xlsx',sheet_name='Queries')
        subfolder = 'SustainDataset'
    elif dataset == 'benchmark2025':
        df = pd.read_excel('../data/text2cypher-2025v1/test.xlsx')
        #update df only with rows with a database access
        df = df[df['url'].notna()].reset_index(drop=True)
        subfolder = 'BenchmarkDataset2025'
    elif dataset == 'benchmark2024':
        df = pd.read_excel('../data/text2cypher-2024v1/test.xlsx')
        #update df only with rows with a database access
        df = df[df['url'].notna()].reset_index(drop=True)
        subfolder = 'BenchmarkDataset2024'
    
    output_dir = f"Outputs_Evaluation/{subfolder}/{model_name}/"
    # Create the directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    results_path_pkl = os.path.join(output_dir, f"{model_name}.pkl")
    results_path_xlsx = os.path.join(output_dir, f"{model_name}.xlsx")
   
    # Check if results file already exists
    if os.path.exists(results_path_xlsx):
        df_results = pd.read_excel(results_path_xlsx)
    else:# Run model on dataset
        df_results = df.copy() # Copy of the df with questions and expected answers, to add the answers of the llm 
        # Now define additional columns in df_results to store LLM results:
        df_results['generated_cypher'] = None 
        df_results['answer'] = None
        df_results['path'] = None           # Use None so we can store a list
        df_results['output'] = None           # Use None so we can store a list
        df_results['cypher_errors_history'] = None           # Use None so we can store a list
        df_results['cypher_errors'] = None           # Use None so we can store a list
        df_results['is_KG_Valid_correct'] = None           # Use None so we can store a list
        df_results['total_tokens'] = None           # Use None so we can store a list
        df_results['completion_tokens'] = None           # Use None so we can store a list
        df_results['prompt_tokens'] = None           # Use None so we can store a list

    config = {'configurable': {'model': model, 'model_name':model_name, 'attempts_w_filtered':5, 'total_attempts':10},'recursion_limit':50}
    
    print(f'Starting evaluation for {model_name} on {dataset} ...')
    # Iterate over each row and compute only if not already done
    for i, row in tqdm(df_results.iterrows(), total=df_results.shape[0]):
        if pd.isna(row['generated_cypher']):
            question = row['Question'] if dataset == 'sustain' else row['question']
            print(i,question)
            
            # Get credentials
            required_cols = {'database_name', 'url', 'username', 'password'}
            if required_cols.issubset(df_results.columns):
                database_url = row['url']
                database_name = row['database_name']
                database_user = row['username']
                database_password = row['password']
            else:
                #its the sustain dataset
                database_url = os.getenv('database_url')
                database_name = os.getenv('database_name')
                database_user = os.getenv('database_user')
                database_password = os.getenv('database_password')
                # activate it for gemma 9b
                # if database_name == 'offshoreleaks':
                #     print(f'Skipping row {i}, to avoid memory error in {database_name}')
                #     continue
                # else:# Configuration 
                inputs = {
                    'question':question,
                    'database_url': database_url,
                    'database_name':database_name,
                    'database_user':database_user,
                    'database_password':database_password
                }
                # torch.cuda.empty_cache() #clear gpu cache
            
                # Run the graph and save outputs
                result =app.invoke(inputs, config =config) #,debug=False
                df_results.at[i, 'generated_cypher'] = result['cypher_statement']
                df_results.at[i, 'output'] = str(result['database_records'])
                df_results.at[i, 'path'] = result['path']
                df_results.at[i, 'cypher_errors'] = result['cypher_errors']
                df_results.at[i, 'cypher_errors_history'] = result['cypher_errors_history']
                df_results.at[i, 'is_KG_Valid_correct'] = result['is_KG_Valid_correct']
                df_results.at[i, 'answer'] = result['answer']
                df_results.at[i, 'total_tokens'] = result['total_tokens']
                df_results.at[i, 'completion_tokens'] = result['completion_tokens']
                df_results.at[i, 'prompt_tokens'] = result['prompt_tokens']
                if ('schema_retriever' in result['path']) and result['is_KG_Valid_correct']:
                    df_results.at[i, 'is_KG_Valid_correct_with_full_schema'] = True
                else:
                    df_results.at[i, 'is_KG_Valid_correct_with_full_schema'] = False
                torch.cuda.empty_cache()#clear gpu cache
                
                # Save results
                df_results.to_pickle(results_path_pkl)
                df_results.to_excel(results_path_xlsx, index=False)# Set index=False to exclude the index column in Excel
        else:
            print(f"Skipping row {i}, already computed.")
            
    return df_results

In [ ]:
model_name = 'llama3_2_3b'
df_results = evaluate_model (model_name, models[model_name](), 'benchmark2025')

Starting evaluation for llama3_2_3b on benchmark2025 ...


  0%|          | 0/2128 [00:00<?, ?it/s]

Skipping row 0, already computed.
Skipping row 1, already computed.
Skipping row 2, already computed.
Skipping row 3, already computed.
Skipping row 4, already computed.
Skipping row 5, already computed.
Skipping row 6, already computed.
Skipping row 7, already computed.
Skipping row 8, already computed.
Skipping row 9, already computed.
Skipping row 10, already computed.
Skipping row 11, already computed.
Skipping row 12, already computed.
Skipping row 13, already computed.
Skipping row 14, already computed.
Skipping row 15, already computed.
Skipping row 16, already computed.
Skipping row 17, already computed.
Skipping row 18, already computed.
Skipping row 19, already computed.
Skipping row 20, already computed.
Skipping row 21, already computed.
Skipping row 22, already computed.
Skipping row 23, already computed.
Skipping row 24, already computed.
Skipping row 25, already computed.
Skipping row 26, already computed.
Skipping row 27, already computed.
Skipping row 28, already compu

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: stripped_off_date)} {position: line: 1, column: 43, offset: 42} for query: 'RETURN exists { MATCH (x:Entity) WHERE (x.stripped_off_date) IS NOT NULL} AS exists'


CyVer Errors from generated query:
- Neo.ClientError.Statement.SyntaxError: Invalid input '}': expected 'WHERE' or ']' (line 1, column 93 (offset: 92))


- UnknownPathWarning: The specified path (:Address)-[:connected_to]->() does not exist in the knowledge graph schema. The path (:Address)-[:connected_to {valid_until: 'YYYY-MM-DD'}]->() in the Cypher statement is wrong.

- Neo.ClientNotification.Statement.UnknownPropertyKeyWarning: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: stripped_off_date)

- InvalidPropertyAccessByLabelWarning: The label Entity does not have the following properties:  country, stripped_off_date.

Cypher Corrector: START
CyVer Evaluator: START
CyVer Evaluation of MATCH (e:Entity {countries: 'Samoa'})
WHERE EXISTS ((e)-[:registered_address {valid_until: 'YYYY-MM-DD'}]->(:Address))
RETURN avg(CAS

...

CyVer Errors from generated query:
- WrongDirectionPathWarning: The specified path (:Genre)-[:IN_GENRE]->(:Movie) has the opposite direction in the knowledge graph schema. The path (g)-[:IN_GENRE {genre: 'Action'}]->(m) in the Cypher statement should be changed to  (g)<-[:IN_GENRE {genre: 'Action'}]-(m).

- Neo.ClientNotification.Statement.UnknownPropertyKeyWarning: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: genre)

- InvalidPropertyAccessByLabelWarning: The label IN_GENRE does not have the following properties:  genre.

Cypher Corrector: START
CyVer Evaluator: START
CyVer Evaluation of MATCH (g:Genre {name: 'Action'})
CREATE (m:Movie {url: 'https://example.com/movie', poster: 'poster.png', runtime: 120, budget: 10000000, revenue: 50000000})
CREATE (m)<-[:IN_GENRE {genre: 'Action'}]-(g):


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: genre)} {position: line: 1, column: 51, offset: 50} for query: 'RETURN exists { MATCH ()-[x:IN_GENRE]-() WHERE (x.genre) IS NOT NULL} AS exists'


CyVer Errors from generated query:
- WrongDirectionPathWarning: The specified path (:Movie)<-[:IN_GENRE]-(:Genre) has the opposite direction in the knowledge graph schema. The path (m)<-[:IN_GENRE {genre: 'Action'}]-(g) in the Cypher statement should be changed to  (m)-[:IN_GENRE {genre: 'Action'}]->(g).

- Neo.ClientNotification.Statement.UnknownPropertyKeyWarning: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: genre)

- InvalidPropertyAccessByLabelWarning: The label IN_GENRE does not have the following properties:  genre.

Cypher Corrector: START
CyVer Evaluator: START
CyVer Evaluation of MATCH (g:Genre {name: 'Action'})
MERGE (m:Movie {url: 'https://example.com/movie', poster: 'poster.png', runtime: 120, budget: 10000000, revenue: 50000000})
MERGE (m)-[:IN_GENRE {genre: 'Action'}]->(g):


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: genre)} {position: line: 1, column: 51, offset: 50} for query: 'RETURN exists { MATCH ()-[x:IN_GENRE]-() WHERE (x.genre) IS NOT NULL} AS exists'


CyVer Errors from generated query:
- Neo.ClientNotification.Statement.UnknownPropertyKeyWarning: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: genre)

- InvalidPropertyAccessByLabelWarning: The label IN_GENRE does not have the following properties:  genre.

Cypher Corrector: START
CyVer Evaluator: START
CyVer Evaluation of MATCH (g:Genre {name: 'Action'})
MERGE (m:Movie {url: 'https://example.com/movie', poster: 'poster.png', runtime: 120, budget: 10000000, revenue: 50000000})
MERGE (m)-[:IN_GENRE]->(g):
Cypher Executor: START
Output Generator: START
2125 How to remove all Movies older than the year 2000? Give me the recommended way of deleting a large volume of data
Schema filtering Agent: START
Cypher Generator: START
CyVer Evaluator: START
CyVer Evaluation of MATCH (m:Movie) WHERE m.year < 2000 DELETE m:
Cypher Exe

# Evaluate with computational metrics

In [2]:
def evaluate_model_usage (model_name, model, dataset):
    ''' 
    Evaluate the selected model on the given dataset, by invoking the graph workflow.
    Save the dataframe with the questions and the generated cypher queries to the Outputs_Evaluation folder.
    Return the dataframe
    '''
    # torch.cuda.empty_cache()#clear gpu cache
    # Get datasets 
    if dataset =='sustain':
        df = pd.read_excel('../data/text2cypher-sustaingraph/sustaintest.xlsx',sheet_name='Queries')
        subfolder = 'SustainDataset'
    elif dataset == 'benchmark2025':
        df = pd.read_excel('../data/text2cypher-2025v1/test.xlsx')
        #update df only with rows with a database access
        df = df[df['url'].notna()].reset_index(drop=True)
        subfolder = 'BenchmarkDataset2025'
    elif dataset == 'benchmark2024':
        df = pd.read_excel('../data/text2cypher-2024v1/test.xlsx')
        #update df only with rows with a database access
        df = df[df['url'].notna()].reset_index(drop=True)
        subfolder = 'BenchmarkDataset2024'
    
    output_dir = f"Outputs_Evaluation/{subfolder}/{model_name}/"
    # Create the directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    results_path_pkl = os.path.join(output_dir, f"{model_name}_usage1.pkl")
    results_path_xlsx = os.path.join(output_dir, f"{model_name}_usage1.xlsx")
   
    # Check if results file already exists
    if os.path.exists(results_path_xlsx):
        df_results = pd.read_excel(results_path_xlsx)
    else:# Run model on dataset
        df_results = df.copy() # Copy of the df with questions and expected answers, to add the answers of the llm 
        # Now define additional columns in df_results to store LLM results:
        df_results['generated_cypher'] = None 
        df_results['answer'] = None
        df_results['path'] = None           # Use None so we can store a list
        df_results['output'] = None           # Use None so we can store a list
        df_results['cypher_errors_history'] = None           # Use None so we can store a list
        df_results['cypher_errors'] = None           # Use None so we can store a list
        df_results['is_KG_Valid_correct'] = None           # Use None so we can store a list
        # timing columns
        df_results['eval_time_s'] = None              # <-- total wall-clock time per query
        # tokens columns
        df_results['total_tokens'] = None           # Use None so we can store a list
        df_results['completion_tokens'] = None           # Use None so we can store a list
        df_results['prompt_tokens'] = None           # Use None so we can store a list
    config = {'configurable': {'model': model, 'model_name':model_name, 'attempts_w_filtered':5, 'total_attempts':10},'recursion_limit':50}
    
    print(f'Starting evaluation for {model_name} on {dataset} ...')
    # Iterate over each row and compute only if not already done
    for i, row in tqdm(df_results.iterrows(), total=df_results.shape[0]):
        if pd.isna(row['generated_cypher']):
            question = row['Question'] if dataset == 'sustain' else row['question']
            print(i,question)
            
            # Get credentials
            required_cols = {'database_name', 'url', 'username', 'password'}
            if required_cols.issubset(df_results.columns):
                database_url = row['url']
                database_name = row['database_name']
                database_user = row['username']
                database_password = row['password']
            else:
                #its the sustain dataset
                database_url = os.getenv('database_url')
                database_name = os.getenv('database_name')
                database_user = os.getenv('database_user')
                database_password = os.getenv('database_password')
                # activate it for gemma 9b
                # if database_name == 'offshoreleaks':
                #     print(f'Skipping row {i}, to avoid memory error in {database_name}')
                #     continue
                # else:# Configuration 
                inputs = {
                    'question':question,
                    'database_url': database_url,
                    'database_name':database_name,
                    'database_user':database_user,
                    'database_password':database_password
                }
                # torch.cuda.empty_cache() #clear gpu cache
            
                # Run the graph and save outputs
                start = time.perf_counter() # start wall-clock timer
                result =app.invoke(inputs, config =config) #,debug=False
                elapsed = time.perf_counter() - start # elapsed wall-clock time
                df_results.at[i, 'eval_time_s'] = round(elapsed, 4)
                df_results.at[i, 'generated_cypher'] = result['cypher_statement']
                df_results.at[i, 'output'] = str(result['database_records'])
                df_results.at[i, 'path'] = result['path']
                df_results.at[i, 'cypher_errors'] = result['cypher_errors']
                df_results.at[i, 'cypher_errors_history'] = result['cypher_errors_history']
                df_results.at[i, 'is_KG_Valid_correct'] = result['is_KG_Valid_correct']
                df_results.at[i, 'answer'] = result['answer']
                df_results.at[i, 'total_tokens'] = result['total_tokens']
                df_results.at[i, 'completion_tokens'] = result['completion_tokens']
                df_results.at[i, 'prompt_tokens'] = result['prompt_tokens']
                if ('schema_retriever' in result['path']) and result['is_KG_Valid_correct']:
                    df_results.at[i, 'is_KG_Valid_correct_with_full_schema'] = True
                else:
                    df_results.at[i, 'is_KG_Valid_correct_with_full_schema'] = False
                torch.cuda.empty_cache()#clear gpu cache
                

                # Save results
                df_results.to_pickle(results_path_pkl)
                df_results.to_excel(results_path_xlsx, index=False)# Set index=False to exclude the index column in Excel
        else:
            print(f"Skipping row {i}, already computed.")
            
    return df_results

In [3]:
model_name = 'gpt_4o' # 'qwen2_5_Coder_32b' #'llama3_2_3b'
df_results = evaluate_model_usage (model_name, models[model_name](), 'sustain') #benchmark2025

Starting evaluation for gpt_4o on sustain ...


  0%|          | 0/52 [00:00<?, ?it/s]

0 What is the title of the goal with code 2?
Schema filtering Agent: START
Cypher Generator: START
CyVer Evaluator: START
CyVer Evaluation of 
MATCH (g:Goal {code: "2"}) RETURN g.title:
Cypher Executor: START
Output Generator: START
1 Which is the EUcode for the geoarea with name Greece ?
Schema filtering Agent: START
Cypher Generator: START
CyVer Evaluator: START
CyVer Evaluation of 
MATCH (g:GeoArea {name: "Greece"}) RETURN g.EUcode:
Cypher Executor: START
Output Generator: START
2 Which is the name of the area that has the EUcode FR?
Schema filtering Agent: START
Cypher Generator: START
CyVer Evaluator: START
CyVer Evaluation of 
MATCH (a:Area {EUcode: 'FR'}) RETURN a.name:
Cypher Executor: START
Output Generator: START
3 Which is the title of case study with code cs_1? 
Schema filtering Agent: START
Cypher Generator: START
CyVer Evaluator: START
CyVer Evaluation of 
MATCH (c:CaseStudy {code: 'cs_1'}) RETURN c.title:
Cypher Executor: START
Output Generator: START
4 Which are the haz

Llama 3 - GPU usage:

92.6 %

90.17 %\

74.4 % 

90

85.03

92.2

75.2

93.8

92.48

90.33

91.6

64

In [ ]:
#In the Sustain dataset, Llama 3 2 3b GPU usage:#NVIDIA L4 24GB
llama3_gpu_extended = [92.6,90.17,74.4 ,90,85.03,92.2,75.2,93.8,92.48,90.33,91.6,64]
mean_usage = sum(llama3_gpu_extended)/len(llama3_gpu_extended)
print(f'Mean GPU usage for Llama 3 2 3b extended: {mean_usage} %')
median_usage = sorted(llama3_gpu_extended)[len(llama3_gpu_extended)//2]
print(f'Median GPU usage for Llama 3 2 3b extended: {median_usage} %')

Mean GPU usage for Llama 3 2 3b extended: 85.98416666666667 %
Median GPU usage for Llama 3 2 3b extended: 90.33 %


In [ ]:
# NVIDIA L4 96GB
qwen_32_coder_gpu_extended = [98.25,
100,97.25,100,98,100,97.5,97.5,97.75,100,100,100,98,100,98,100,98,100,100,96.75,97.5,100,100,97.75,99,100,100,100,97.75,98.5,98,100,96.75,96.5]
mean_usage = sum(qwen_32_coder_gpu_extended)/len(qwen_32_coder_gpu_extended)
print(f'Mean GPU usage for Qwen coder 32b extended: {mean_usage} %')
median_usage = sorted(qwen_32_coder_gpu_extended)[len(qwen_32_coder_gpu_extended)//2]
print(f'Median GPU usage for Qwen coder 32b extended: {median_usage} %')

Mean GPU usage for Qwen coder 32b extended: 98.78676470588235 %
Median GPU usage for Qwen coder 32b extended: 99 %
